# ZA Demand Import Export Model Inputs

Module 06 validation notebook. It reads the generated model-input artifacts and checks demand, imports, exports, Other RE, and spatial attachment weights.

In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

root = Path.cwd()
while root != root.parent and not (root / 'Snakefile').exists():
    root = root.parent
if not (root / 'Snakefile').exists():
    raise RuntimeError('Could not find repository root containing Snakefile')

paths = {
    'demand': root / 'data/za_validation/za_2023_demand_profile.csv',
    'gegis': root / 'data/ssp2-2.6/2030/era5_2023_custom/Africa.csv',
    'imports_exports': root / 'data/za_validation/za_2023_import_export_timeseries.csv',
    'other_re': root / 'data/za_validation/za_2023_other_re_timeseries.csv',
    'load_weights': root / 'data/za_audit/za_2023_load_allocation_weights.csv',
    'ie_attachment': root / 'data/za_audit/za_2023_import_export_attachment.csv',
    'other_re_attachment': root / 'data/za_audit/za_2023_other_re_attachment.csv',
    'rsa_comparison': root / 'data/za_audit/pypsa_rsa_gva_pop_load_weight_comparison.csv',
}
for key, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing {key}: {path}')
print(root)


In [ ]:

demand = pd.read_csv(paths['demand'], parse_dates=['time'])
gegis = pd.read_csv(paths['gegis'], sep=';', parse_dates=['time'])
ie = pd.read_csv(paths['imports_exports'], parse_dates=['time'])
other_re = pd.read_csv(paths['other_re'], parse_dates=['time'])
load_weights = pd.read_csv(paths['load_weights'])
ie_attachment = pd.read_csv(paths['ie_attachment'])
other_re_attachment = pd.read_csv(paths['other_re_attachment'])
rsa_comparison = pd.read_csv(paths['rsa_comparison'])

summary = pd.DataFrame([
    {'artifact': 'demand', 'rows': len(demand), 'start': demand.time.min(), 'end': demand.time.max()},
    {'artifact': 'gegis', 'rows': len(gegis), 'start': gegis.time.min(), 'end': gegis.time.max()},
    {'artifact': 'imports_exports', 'rows': len(ie), 'start': ie.time.min(), 'end': ie.time.max()},
    {'artifact': 'other_re', 'rows': len(other_re), 'start': other_re.time.min(), 'end': other_re.time.max()},
])
summary


In [ ]:

expected_rows = 8760
for name, frame in [('demand', demand), ('gegis', gegis), ('imports_exports', ie), ('other_re', other_re)]:
    assert len(frame) == expected_rows, f'{name} has {len(frame)} rows'
    assert frame.time.min() == pd.Timestamp('2023-01-01 00:00:00')
    assert frame.time.max() == pd.Timestamp('2023-12-31 23:00:00')
print('All hourly artifacts have 8760 aligned rows.')


## Demand

In [ ]:

demand_twh = demand['rsa_contracted_demand_mw'].sum() / 1e6
peak = demand['rsa_contracted_demand_mw'].max()
low = demand['rsa_contracted_demand_mw'].min()
pd.DataFrame([{'annual_twh': demand_twh, 'peak_mw': peak, 'minimum_mw': low}])


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
demand.set_index('time')['rsa_contracted_demand_mw'].plot(ax=axes[0], lw=0.7, color='#1f77b4')
axes[0].set_title('RSA Contracted Demand, 2023')
axes[0].set_ylabel('MW')
axes[0].set_xlabel('')

duration = demand['rsa_contracted_demand_mw'].sort_values(ascending=False).reset_index(drop=True)
duration.plot(ax=axes[1], lw=1.0, color='#2ca02c')
axes[1].set_title('Demand Duration Curve')
axes[1].set_ylabel('MW')
axes[1].set_xlabel('hour rank')
fig.tight_layout()
plt.show()


## Imports And Exports

In [ ]:

ie_summary = pd.DataFrame([
    {'series': 'gross imports', 'annual_twh': ie['international_imports_mw'].sum() / 1e6},
    {'series': 'gross exports', 'annual_twh': ie['international_exports_mw'].sum() / 1e6},
    {'series': 'net import', 'annual_twh': ie['net_import_mw'].sum() / 1e6},
])
ie_summary


In [ ]:

fig, ax = plt.subplots(figsize=(11, 4))
ie.set_index('time')[['international_imports_mw', 'international_exports_mw']].plot(ax=ax, lw=0.7)
ax.set_title('Gross International Imports And Exports')
ax.set_ylabel('MW')
ax.set_xlabel('')
fig.tight_layout()
plt.show()


## Other RE

In [ ]:

other_re_summary = pd.DataFrame([{
    'annual_twh': other_re['other_re_mw'].sum() / 1e6,
    'p_nom_mw': other_re['p_nom_mw'].iloc[0],
    'max_p_max_pu': other_re['p_max_pu'].max(),
    'clipped_hours': int(other_re['was_clipped'].sum()),
    'p_min_pu_unique': sorted(other_re['p_min_pu'].unique()),
}])
other_re_summary


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
other_re.set_index('time')['other_re_mw'].plot(ax=axes[0], lw=0.7, color='#88c057')
axes[0].set_title('Other RE Output')
axes[0].set_ylabel('MW')
axes[0].set_xlabel('')
other_re.set_index('time')['p_max_pu'].plot(ax=axes[1], lw=0.7, color='#4d9221')
axes[1].set_title('Other RE p_max_pu')
axes[1].set_ylabel('per unit')
axes[1].set_xlabel('')
fig.tight_layout()
plt.show()


## Attachment Weights

In [ ]:

def weight_sums(frame):
    return frame.groupby(['layer_key', 'attachment_type', 'source_id'], as_index=False)['weight'].sum()

checks = pd.concat([
    weight_sums(load_weights).assign(table='load_weights'),
    weight_sums(ie_attachment).assign(table='import_export_attachment'),
    weight_sums(other_re_attachment).assign(table='other_re_attachment'),
], ignore_index=True)
checks['abs_error'] = (checks['weight'] - 1.0).abs()
assert checks['abs_error'].max() < 1e-9
checks


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
plot_weights = load_weights.loc[load_weights['layer_key'].isin([10, 34])].copy()
plot_weights['label'] = plot_weights['layer_key'].astype(str) + ': ' + plot_weights['target_region_id'].astype(str)
plot_weights.sort_values(['layer_key', 'weight'], ascending=[True, False]).head(20).plot.bar(
    x='label', y='weight', ax=ax, color='#4c78a8', legend=False
)
ax.set_title('Largest Demand Allocation Weights')
ax.set_ylabel('weight')
ax.set_xlabel('')
plt.xticks(rotation=75, ha='right')
fig.tight_layout()
plt.show()


## PyPSA-RSA GVA/POP Diagnostic

In [ ]:

rsa_comparison['status'].value_counts().rename_axis('status').reset_index(name='rows')


In [ ]:

rsa_comparison.head(12)
